# KUNCI JAWABAN — Tugas 2: Laporan Praktikum Pembersihan Data Transaksi
**Business Statistics and Data Visualization · Pertemuan 2 · khusus dosen**

Dataset: `transaksi_tugas2_kotor.csv` (fiktif, Juli–September 2026). Angka acuan di notebook ini dapat dipakai untuk memeriksa jawaban mahasiswa. Strategi lain tetap dapat diterima bila alasannya logis dan konsisten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

tugas_raw = pd.read_csv("transaksi_tugas2_kotor.csv")
df = tugas_raw.copy()
log = []
def catat(langkah, ket, sebelum, sesudah, terdampak=None):
    log.append({"langkah": langkah, "keterangan": ket, "baris_sebelum": sebelum, "baris_sesudah": sesudah,
                "baris_terdampak": sebelum - sesudah if terdampak is None else terdampak})

## Soal 1 — Profil data (15%)

In [ ]:
print("Ukuran:", df.shape)
profil = pd.DataFrame({"tipe": df.dtypes.astype(str), "kosong": df.isna().sum(),
                       "persen_kosong": (df.isna().mean() * 100).round(1), "nilai_unik": df.nunique()})
profil

**Acuan jawaban:** `harga` bertipe teks karena sebagian berformat `Rp 150.000`; `tanggal` bertipe teks karena ada dua format (`YYYY-MM-DD` dan `DD/MM/YYYY`). Kolom kosong: `rating`, `kota`, `metode_bayar`. Mahasiswa juga sebaiknya menyebut bahwa `kota` dan `metode_bayar` memiliki terlalu banyak nilai unik (label tidak seragam).

## Soal 2 — Duplikat & label (20%)

In [ ]:
n0 = len(df)
print("Duplikat persis:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
catat("Hapus duplikat", "baris identik", n0, len(df))

print("\nSEBELUM — kota:", sorted(df["kota"].dropna().unique()))
print("SEBELUM — metode_bayar:", sorted(df["metode_bayar"].dropna().unique()))
peta_kota = {"Mks": "Makassar", "Jkt": "Jakarta", "Dki Jakarta": "Jakarta", "Sby": "Surabaya", "Bdg": "Bandung"}
df["kota"] = df["kota"].str.strip().str.title().replace(peta_kota)
peta_bayar = {"ewallet": "E-Wallet", "e-wallet": "E-Wallet", "cod": "COD", "transfer": "Transfer Bank",
              "transfer bank": "Transfer Bank", "kartu kredit": "Kartu Kredit", "cc": "Kartu Kredit"}
df["metode_bayar"] = df["metode_bayar"].str.strip().str.lower().replace(peta_bayar)
print("\nSESUDAH — kota:", sorted(df["kota"].dropna().unique()))
print("SESUDAH — metode_bayar:", sorted(df["metode_bayar"].dropna().unique()))

## Soal 3 — Tipe data & missing value (20%)

In [ ]:
df["harga"] = df["harga"].astype(str).str.replace(r"[^0-9]", "", regex=True).astype("int64")
t1 = pd.to_datetime(df["tanggal"], format="%Y-%m-%d", errors="coerce")
t2 = pd.to_datetime(df["tanggal"], format="%d/%m/%Y", errors="coerce")
df["tanggal"] = t1.fillna(t2)
print("Tanggal gagal diparsing:", df["tanggal"].isna().sum(), "| rentang:", df["tanggal"].min().date(), "–", df["tanggal"].max().date())

print("\nData kosong sebelum ditangani:\n", df.isna().sum()[lambda s: s > 0].to_string())
df["kota"] = df["kota"].fillna("Tidak diketahui")
df["metode_bayar"] = df["metode_bayar"].fillna("Tidak diketahui")
df["ada_rating"] = df["rating"].notna()
print("\nData kosong sesudah:\n", df.isna().sum()[lambda s: s > 0].to_string())

**Acuan alasan:** `kota` dan `metode_bayar` kosong < 5% dan bertipe kategori, jadi diisi "Tidak diketahui" agar omzetnya tidak hilang. `rating` kosong karena pembeli tidak memberi ulasan, jadi **tidak diimputasi** dengan rata-rata karena akan membuat rating tampak lebih seragam dari kenyataannya. Jawaban yang mengimputasi rating dengan median dan menandainya dengan *flag* tetap dapat diterima bila dijelaskan.

## Soal 4 — Nilai tidak valid & outlier (25%)

In [ ]:
salah_rating = df["rating"].notna() & ~df["rating"].between(1, 5)
print("Rating tidak valid:", salah_rating.sum(), sorted(df.loc[salah_rating, "rating"].unique()))
df.loc[salah_rating, "rating"] = np.nan
df["ada_rating"] = df["rating"].notna()
log.append({"langkah": "Rating tidak valid → kosong", "keterangan": "rating di luar 1–5", "baris_sebelum": len(df),
            "baris_sesudah": len(df), "baris_terdampak": int(salah_rating.sum())})

n0 = len(df)
print("Jumlah ≤ 0:", (df["jumlah"] <= 0).sum())
df = df[df["jumlah"] >= 1].reset_index(drop=True)
catat("Hapus jumlah tidak valid", "jumlah ≤ 0", n0, len(df))

In [ ]:
def batas_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return pd.Series({"Q1": q1, "Q3": q3, "batas_atas": q3 + 1.5 * iqr})

batas = df.groupby("kategori")["harga"].apply(batas_iqr).unstack()
df = df.merge(batas[["batas_atas"]], left_on="kategori", right_index=True)
df["outlier_iqr"] = df["harga"] > df["batas_atas"]
print("Outlier harga (IQR per kategori):\n", df.groupby("kategori")["outlier_iqr"].sum().to_string())

df["rasio_median"] = df["harga"] / df.groupby("kategori")["harga"].transform("median")
df.sort_values("rasio_median", ascending=False)[["order_id", "kategori", "harga", "rasio_median"]].head(10)

In [ ]:
typo = (df["rasio_median"] > 5) & df["kategori"].isin(["Fashion", "Kecantikan", "Makanan"])
df.loc[typo, "harga"] = df.loc[typo, "harga"] // 10
log.append({"langkah": "Koreksi harga salah ketik", "keterangan": "> 5× median kategori, dibagi 10", "baris_sebelum": len(df),
            "baris_sesudah": len(df), "baris_terdampak": int(typo.sum())})
df["grosir"] = df["jumlah"] > 20
print("Harga dikoreksi:", typo.sum(), "| transaksi grosir dipertahankan:", df["grosir"].sum())
df = df.drop(columns=["batas_atas", "rasio_median"])

**Acuan:** sebagian besar outlier IQR adalah harga mahal yang **wajar** (terutama Elektronik dan sebagian Fashion), jadi dipertahankan. Hanya harga > 5× median pada kategori berharga rendah yang dikoreksi sebagai salah ketik. Pembelian grosir (> 20 unit) adalah **transaksi nyata**: dipertahankan dan ditandai. Nilai poin penuh diberikan bila mahasiswa **membedakan** outlier yang merupakan kesalahan dari outlier yang nyata, bukan sekadar menghapus semua outlier.

## Soal 5 — EDA & insight (20%)

In [ ]:
df["total"] = df["harga"] * df["jumlah"] + df["ongkir"]
df["bulan"] = df["tanggal"].dt.month.map({7: "Juli", 8: "Agustus", 9: "September"})
print(f"Transaksi bersih: {len(df):,} | omzet (termasuk ongkir): Rp{df['total'].sum():,.0f}")
ring = df.groupby("kategori").agg(transaksi=("order_id", "count"), omzet=("total", "sum"), rating=("rating", "mean")).sort_values("omzet", ascending=False)
ring["porsi_%"] = (ring["omzet"] / ring["omzet"].sum() * 100).round(1)
ring

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
(ring["omzet"] / 1e6).sort_values().plot.barh(ax=axes[0], color="#0FA3B1", title="Omzet per kategori (juta Rp)")
bulanan = df.groupby("bulan", sort=False)["total"].sum().reindex(["Juli", "Agustus", "September"]) / 1e6
bulanan.plot(ax=axes[1], marker="o", color="#EE6C4D", title="Omzet per bulan (juta Rp)")
df.groupby("metode_bayar")["rating"].mean().drop("Tidak diketahui").sort_values().plot.barh(ax=axes[2], color="#F7B32B", title="Rating rata-rata per metode bayar", xlim=(3, 4.6))
plt.tight_layout(); plt.show()
bulanan

In [ ]:
rb = df.groupby("metode_bayar")["rating"].mean().drop("Tidak diketahui")
ong = df.groupby("kota")["ongkir"].mean()
print(f"Insight 1 — kategori omzet terbesar: {ring.index[0]} ({ring['porsi_%'].iloc[0]}% omzet)")
print(f"Insight 2 — rating COD {rb['COD']:.2f} vs non-COD {rb.drop('COD').mean():.2f}")
print(f"Insight 3 — ongkir rata-rata Jakarta Rp{ong['Jakarta']:,.0f} vs kota lain Rp{ong.drop(['Jakarta','Tidak diketahui'], errors='ignore').mean():,.0f}")

**Contoh insight acuan** (lihat angka dari sel di atas):
1. Elektronik menyumbang porsi omzet terbesar meski jumlah transaksinya paling sedikit → fokuskan promosi bernilai tinggi (cicilan, garansi) pada kategori ini.
2. Pembeli COD memberi rating lebih rendah daripada pembeli non-COD → periksa pengalaman pengiriman COD; uji insentif pembayaran non-tunai.
3. Ongkir rata-rata di luar Jakarta hampir dua kali ongkir Jakarta → subsidi ongkir untuk kota di luar Jakarta berpotensi menaikkan konversi.

In [ ]:
log_df = pd.DataFrame(log)
print(f"Mentah {len(tugas_raw):,} baris → bersih {len(df):,} baris")
log_df